# Линейные модели: регрессия и классификация

Линейные модели — один из фундаментальных инструментов машинного обучения. Несмотря на кажущуюся простоту, они:

- хорошо интерпретируемы — можно посмотреть на коэффициенты и понять, что влияет на результат;
- быстро обучаются даже на миллионах объектов;
- служат сильным baseline'ом для более сложных методов;
- лежат в основе нейросетей.

На этом занятии мы разберём линейную и логистическую регрессию, регуляризацию, а также кривые валидации и обучения — важнейшие инструменты диагностики модели.

### Содержание
1. [Линейная регрессия и МНК](#1.-Линейная-регрессия-и-МНК)
2. [Метод максимального правдоподобия](#2.-Метод-максимального-правдоподобия)
3. [Разложение ошибки: смещение и разброс](#3.-Разложение-ошибки:-смещение-и-разброс)
4. [Регуляризация](#4.-Регуляризация)
5. [Логистическая регрессия](#5.-Логистическая-регрессия)
6. [Регуляризация логистической регрессии: наглядный пример](#6.-Регуляризация-логистической-регрессии:-наглядный-пример)
7. [Где логистическая регрессия работает хорошо, а где — нет](#7.-Где-логистическая-регрессия-работает-хорошо,-а-где-—-нет)
8. [Кривые валидации и обучения](#8.-Кривые-валидации-и-обучения)
9. [Практика](#9.-Практика)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()
from matplotlib import pyplot as plt

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, SGDClassifier
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, learning_curve, validation_curve)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer

%config InlineBackend.figure_format = 'svg'

---
## 1. Линейная регрессия и МНК

**Задача регрессии**: по набору признаков $\mathbf{x} = (x_1, x_2, \ldots, x_d)$ предсказать вещественное число $y$ (цену квартиры, объём потребления газа, температуру).

Линейная модель делает простое предположение: зависимость между признаками и целевой переменной — линейная:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots + w_d x_d = \mathbf{w}^T \mathbf{x}$$

где $w_0$ — свободный член (bias), $w_1, \ldots, w_d$ — веса признаков.

### Метод наименьших квадратов (МНК)

Самый классический способ подобрать веса — **минимизировать сумму квадратов ошибок** (Mean Squared Error, MSE):

$$\text{MSE}(\mathbf{X}, \mathbf{y}, \mathbf{w}) = \frac{1}{\ell}\sum_{i=1}^{\ell}(y_i - \mathbf{w}^T \mathbf{x}_i)^2$$

Эта функция выпуклая, и минимум достигается аналитически:

$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Это называется **нормальным уравнением МНК**.

**Почему квадрат, а не просто модуль?** Квадратичная функция дифференцируема всюду и имеет красивое аналитическое решение. При минимизации абсолютного отклонения аналитического решения нет, нужны итерационные методы.

### Оценки и несмещённость

Оценка $\hat{w}_i$ называется **несмещённой**, если её математическое ожидание равно истинному значению параметра:

$$\mathbb{E}[\hat{w}_i] = w_i$$

MНК-оценки несмещённые (при выполнении условий теоремы Гаусса-Маркова) и обладают наименьшей дисперсией среди всех линейных несмещённых оценок — это называется **теоремой Гаусса-Маркова** (BLUE: Best Linear Unbiased Estimator).

---
## 2. Метод максимального правдоподобия

Почему именно MSE? Есть более глубокое обоснование из вероятностной теории.

Предположим, что истинная зависимость такова:

$$y = \mathbf{w}^T \mathbf{x} + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

То есть ответ — это линейная функция признаков плюс **гауссовский шум**. Тогда:

$$P(y_i \mid \mathbf{x}_i, \mathbf{w}) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \mathbf{w}^T \mathbf{x}_i)^2}{2\sigma^2}\right)$$

Метод максимального правдоподобия (MLE) говорит: найдём $\mathbf{w}$, которое делает наблюдённые данные наиболее вероятными. Логарифм правдоподобия:

$$\log P(\mathbf{y} \mid \mathbf{X}, \mathbf{w}) = -\frac{\ell}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{\ell}(y_i - \mathbf{w}^T \mathbf{x}_i)^2$$

**Вывод:** максимизация log-правдоподобия при гауссовском шуме эквивалентна минимизации MSE. МНК — это не просто «удобная формула», а следствие предположения о нормальности ошибок.

---
## 3. Разложение ошибки: смещение и разброс

Это один из ключевых концептов в ML, применимый к любому алгоритму.

Пусть истинная зависимость: $y = f(\mathbf{x}) + \varepsilon$, где $\mathbb{E}[\varepsilon] = 0$, $\text{Var}(\varepsilon) = \sigma^2$.

Ожидаемая ошибка модели $\hat{f}$ на новом объекте $\mathbf{x}$ раскладывается в сумму трёх составляющих:

$$\mathbb{E}\left[(y - \hat{f})^2\right] = \underbrace{\left(\mathbb{E}[\hat{f}] - f\right)^2}_{\text{Смещение}^2} + \underbrace{\mathbb{E}\left[\left(\hat{f} - \mathbb{E}[\hat{f}]\right)^2\right]}_{\text{Разброс}} + \underbrace{\sigma^2}_{\text{Шум}}$$

| Составляющая | Что означает | Как влияет |
|---|---|---|
| **Смещение (Bias)** | Насколько модель систематически ошибается | Слишком простая модель → большое смещение → **недообучение** |
| **Разброс (Variance)** | Насколько модель чувствительна к конкретной выборке | Слишком сложная модель → большой разброс → **переобучение** |
| **Шум** | Неустранимая случайная ошибка в данных | Нельзя уменьшить |

### Иллюстрация компромисса

In [ ]:
np.random.seed(42)

# Истинная функция: sin(x)
x_true = np.linspace(0, 3 * np.pi, 300)
y_true = np.sin(x_true)

# Несколько обучающих выборок — имитируем разброс оценок модели
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, degree, title in zip(axes, [1, 15], ["Линейная модель (высокое смещение)", "Полином 15-й степени (высокий разброс)"]):
    ax.plot(x_true, y_true, 'k-', lw=2, label='Истинная функция', alpha=0.7)
    for _ in range(8):
        x_sample = np.random.uniform(0, 3 * np.pi, 20)
        y_sample = np.sin(x_sample) + 0.3 * np.random.randn(20)
        coeffs = np.polyfit(x_sample, y_sample, degree)
        y_fit = np.polyval(coeffs, x_true)
        # ограничиваем для читаемости
        y_fit = np.clip(y_fit, -3, 3)
        ax.plot(x_true, y_fit, alpha=0.4, lw=1)
    ax.set_title(title)
    ax.set_ylim(-3, 3)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()

**Что видно:**
- Линейная модель (слева) — все 8 кривых похожи (маленький разброс), но все систематически далеки от истины (большое смещение).
- Полином 15-й степени (справа) — кривые «прыгают» от выборки к выборке (большой разброс), хотя в среднем могли бы попасть в цель.

Цель обучения — найти компромисс между смещением и разбросом.

---
## 4. Регуляризация

Что делать, если матрица $\mathbf{X}^T \mathbf{X}$ необратима (мультиколлинеарность, $d > \ell$)? Или модель переобучается?

Ответ — **регуляризация**: добавим штраф за слишком большие веса к функции потерь.

### Ridge-регрессия (L2)

$$\mathcal{L}_{\text{ridge}}(\mathbf{w}) = \text{MSE} + \alpha \|\mathbf{w}\|_2^2 = \frac{1}{\ell}\sum_i(y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \alpha \sum_j w_j^2$$

Параметр $\alpha > 0$ — сила регуляризации. Решение стабилизируется:

$$\mathbf{w}_{\text{ridge}} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$$

### Lasso-регрессия (L1)

$$\mathcal{L}_{\text{lasso}}(\mathbf{w}) = \text{MSE} + \alpha \|\mathbf{w}\|_1 = \frac{1}{\ell}\sum_i(y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \alpha \sum_j |w_j|$$

Lasso обнуляет многие веса — делает **отбор признаков** автоматически.

### Сравнение

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Штраф | $\sum w_j^2$ | $\sum |w_j|$ |
| Эффект | Уменьшает все веса | Обнуляет часть весов |
| Отбор признаков | Нет | Да |
| Аналитическое решение | Есть | Нет |

> **Важно:** перед применением регуляризации признаки нужно масштабировать (`StandardScaler`), иначе штраф будет несправедливо велик для признаков с большими абсолютными значениями.

---
## 5. Логистическая регрессия

Теперь перейдём к задаче классификации. Несмотря на название, логистическая регрессия — это **классификатор**, а не регрессор.

### Идея линейного классификатора

Если объекты двух классов можно разделить гиперплоскостью в пространстве признаков, задача называется **линейно разделимой**.

Базовый линейный классификатор:
$$\hat{y} = \text{sign}(\mathbf{w}^T \mathbf{x})$$

Но знак — это жёсткое решение. Гораздо полезнее знать **вероятность** принадлежности к классу.

### Сигмоида и log-odds

Логистическая регрессия предсказывает вероятность класса "+1":

$$p_+ = P(y=1 \mid \mathbf{x}, \mathbf{w}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}}$$

Функция $\sigma(z)$ называется **сигмоидой** — она «сжимает» любое вещественное число в интервал $(0, 1)$.

In [ ]:
z = np.linspace(-6, 6, 200)
sigma = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 4))
plt.plot(z, sigma, lw=2, color='steelblue')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='p = 0.5')
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.xlabel('z = $\\mathbf{w}^T \\mathbf{x}$')
plt.ylabel('$\\sigma(z)$')
plt.title('Сигмоидная функция')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-0.05, 1.05)

Логарифм отношения шансов (log-odds или логит) — линейная функция признаков:

$$\log \frac{p_+}{1 - p_+} = \mathbf{w}^T \mathbf{x}$$

Отсюда название: логистическая регрессия моделирует **логит** вероятности как линейную функцию признаков.

### Функция потерь

Обучение логистической регрессии — это максимизация правдоподобия. Она эквивалентна минимизации **логистической функции потерь**:

$$\mathcal{L}_{\log}(\mathbf{X}, \mathbf{y}, \mathbf{w}) = \sum_{i=1}^{\ell} \log(1 + \exp(-y_i \mathbf{w}^T \mathbf{x}_i))$$

где $y_i \in \{-1, +1\}$.

Величина $M_i = y_i \mathbf{w}^T \mathbf{x}_i$ называется **отступом** (margin): чем он больше, тем увереннее классификатор в правильном ответе. Логистические потери штрафуют за малые или отрицательные отступы.

### Регуляризация в sklearn

В `LogisticRegression` из sklearn регуляризация задаётся параметром **C** — обратным коэффициентом регуляризации:

$$\mathcal{L} + \frac{1}{C} \|\mathbf{w}\|^2$$

- **Большой C** → слабая регуляризация → более сложная модель (риск переобучения)
- **Маленький C** → сильная регуляризация → более простая модель (риск недообучения)

---
## 6. Регуляризация логистической регрессии: наглядный пример

Посмотрим, как регуляризация влияет на форму разделяющей границы. Будем работать с данными о тестировании микросхем из курса Andrew Ng: 118 чипов, два результата контроля качества, целевая переменная — пригоден ли чип.

Оранжевые точки — бракованные чипы, синие — рабочие.

In [ ]:
import pandas as pd

data = pd.read_csv(
    "../../data/microchip_tests.txt", header=None, names=("test1", "test2", "released")
)

X = data.iloc[:, :2].values
y = data.iloc[:, 2].values

plt.figure(figsize=(7, 5))
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="steelblue", label="Рабочий", edgecolors='white')
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", label="Бракованный", edgecolors='white')
plt.xlabel("Тест 1")
plt.ylabel("Тест 2")
plt.title("Данные тестирования микросхем")
plt.legend()
plt.grid(True, alpha=0.3)

Классы явно нелинейно разделены. Чтобы линейная модель смогла справиться с этим, добавим **полиномиальные признаки** степени 7. Например, для $d=3$ и двух признаков $x_1, x_2$ добавляем:

$$1, x_1, x_2, x_1^2, x_1 x_2, x_2^2, x_1^3, x_1^2 x_2, x_1 x_2^2, x_2^3$$

Для $d=7$ получается 36 признаков.

In [ ]:
poly = PolynomialFeatures(degree=7)
X_poly = poly.fit_transform(X)
print(f"Исходных признаков: {X.shape[1]}, после полиномиального расширения: {X_poly.shape[1]}")

In [ ]:
def plot_boundary(clf, X, y, grid_step=0.01, poly_featurizer=None):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, grid_step),
        np.arange(y_min, y_max, grid_step)
    )
    Z_in = np.c_[xx.ravel(), yy.ravel()]
    if poly_featurizer:
        Z_in = poly_featurizer.transform(Z_in)
    Z = clf.predict_proba(Z_in)[:, 1].reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)

Обучим три модели с разными значениями C и посмотрим на разделяющую границу:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, C, title in zip(
    axes,
    [1e-2, 1, 1e4],
    ["C=0.01 (сильная регуляризация)", "C=1 (умеренная)", "C=10000 (почти без регуляризации)"]
):
    logit = LogisticRegression(C=C, random_state=17, max_iter=1000)
    logit.fit(X_poly, y)
    plt.sca(ax)
    plot_boundary(logit, X, y, grid_step=0.01, poly_featurizer=poly)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="steelblue", edgecolors='white', s=40)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="orange", edgecolors='white', s=40)
    acc = logit.score(X_poly, y)
    ax.set_title(f"{title}\nТочность: {acc:.2f}")
    ax.set_xlabel("Тест 1")
    ax.set_ylabel("Тест 2")

plt.tight_layout()

**Выводы:**
- **C=0.01** — сильная регуляризация, граница слишком гладкая, модель недообучена.
- **C=1** — хороший баланс: граница разумной формы, модель хорошо обобщается.
- **C=10000** — почти без регуляризации, граница «запомнила» обучающие данные, переобучение.

### Автоматический подбор C через кросс-валидацию

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)
c_values = np.logspace(-2, 3, 200)

logit_searcher = LogisticRegressionCV(Cs=c_values, cv=skf, n_jobs=-1, max_iter=1000)
logit_searcher.fit(X_poly, y)

print(f"Оптимальное C = {logit_searcher.C_[0]:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(c_values, np.mean(logit_searcher.scores_[1], axis=0), color='steelblue', lw=2)
plt.xlabel("C")
plt.ylabel("Средняя точность (CV)")
plt.title("Зависимость качества от параметра регуляризации C")
plt.grid(True, alpha=0.3)

# Крупнее — окрестность оптимума
plt.figure(figsize=(8, 4))
plt.plot(c_values, np.mean(logit_searcher.scores_[1], axis=0), color='steelblue', lw=2)
plt.xlabel("C")
plt.ylabel("Средняя точность (CV)")
plt.title("Окрестность оптимального C")
plt.xlim(0, 10)
plt.grid(True, alpha=0.3)

---
## 7. Где логистическая регрессия работает хорошо, а где — нет

### 7.1 Анализ тональности отзывов IMDB

Логистическая регрессия отлично подходит для задач с **разреженными высокоразмерными признаками** — например, для классификации текстов методом «мешок слов» (Bag of Words).

Каждый отзыв превращается в вектор частот слов: всего около 74 тысяч уникальных слов, и для каждого отзыва большинство из них равны нулю (разреженная матрица).

Датасет: 25000 обучающих и 25000 тестовых отзывов с IMDB, равномерно разделённых на положительные и отрицательные.

In [ ]:
import os
from sklearn.datasets import load_files

PATH_TO_IMDB = "../../data/aclImdb"

if os.path.exists(PATH_TO_IMDB):
    reviews_train = load_files(os.path.join(PATH_TO_IMDB, "train"), categories=["pos", "neg"])
    text_train, y_train = reviews_train.data, reviews_train.target
    reviews_test = load_files(os.path.join(PATH_TO_IMDB, "test"), categories=["pos", "neg"])
    text_test, y_test = reviews_test.data, reviews_test.target
    print(f"Обучающих отзывов: {len(text_train)}, тестовых: {len(text_test)}")
else:
    print("Датасет IMDB не найден — пропускаем этот раздел.")
    print("Скачайте его по ссылке: http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz")
    print("и распакуйте в папку ../../data/")

In [ ]:
if os.path.exists(PATH_TO_IMDB):
    # Bag of Words → матрица признаков
    cv_text = CountVectorizer()
    X_train_text = cv_text.fit_transform(text_train)
    X_test_text  = cv_text.transform(text_test)
    print(f"Размер матрицы признаков: {X_train_text.shape}")
    print(f"Плотность матрицы: {X_train_text.nnz / (X_train_text.shape[0] * X_train_text.shape[1]):.5f}")

In [ ]:
if os.path.exists(PATH_TO_IMDB):
    logit_text = LogisticRegression(solver="lbfgs", n_jobs=-1, random_state=7, max_iter=1000)
    logit_text.fit(X_train_text, y_train)
    print(f"Точность на обучении: {logit_text.score(X_train_text, y_train):.3f}")
    print(f"Точность на тесте:    {logit_text.score(X_test_text,  y_test):.3f}")

In [ ]:
def visualize_coefficients(classifier, feature_names, n_top=20):
    coef = classifier.coef_.ravel()
    top_pos = np.argsort(coef)[-n_top:]
    top_neg = np.argsort(coef)[:n_top]
    idx = np.hstack([top_neg, top_pos])

    plt.figure(figsize=(12, 5))
    colors = ['orange'] * n_top + ['steelblue'] * n_top
    plt.bar(range(2 * n_top), coef[idx], color=colors)
    plt.xticks(range(2 * n_top), [feature_names[i] for i in idx], rotation=60, ha='right', fontsize=9)
    plt.axhline(0, color='black', linewidth=0.8)
    plt.title('Топ слов: негативные (оранжевые) и позитивные (синие) для классификации')
    plt.tight_layout()

if os.path.exists(PATH_TO_IMDB):
    visualize_coefficients(logit_text, cv_text.get_feature_names_out())

Коэффициенты модели — это и есть интерпретация: слова с большими положительными весами ассоциированы с позитивными отзывами, с большими отрицательными — с негативными.

### 7.2 XOR-проблема: где логистическая регрессия не справляется

Линейный классификатор может построить только плоскую границу — гиперплоскость. Есть задачи, где это принципиально недостаточно.

Классический пример — задача XOR: класс объекта определяется исключительно чётностью знаков его координат.

In [ ]:
rng = np.random.RandomState(0)
X_xor = rng.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0, X_xor[:, 1] > 0)

plt.figure(figsize=(5, 5))
plt.scatter(X_xor[:, 0], X_xor[:, 1], s=30, c=y_xor, cmap=plt.cm.Paired)
plt.title('XOR-задача: прямой линией классы не разделить')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.grid(True, alpha=0.3)

In [ ]:
def plot_xor_boundary(clf, X, y, title):
    xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
    clf.fit(X, y)
    Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired, edgecolors='white')
    plt.title(title)
    plt.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plt.sca(axes[0])
plot_xor_boundary(LogisticRegression(solver='lbfgs'), X_xor, y_xor,
                  'Логистическая регрессия\n(линейная граница)')

logit_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("logit", LogisticRegression(solver="lbfgs")),
])
plt.sca(axes[1])
plot_xor_boundary(logit_poly, X_xor, y_xor,
                  'Логистическая регрессия + полиномиальные признаки\n(нелинейная граница)')

plt.tight_layout()

**Что произошло во втором случае?** Добавляя полиномиальные признаки $x_1^2, x_1 x_2, x_2^2$, мы переходим в 6-мерное пространство. В нём классы линейно разделимы — и логистическая регрессия строит там гиперплоскость. При проекции обратно в 2D она выглядит как кривая.

**Вывод:** логистическая регрессия сама по себе строит только линейную границу. С полиномиальными признаками — нелинейную. Это требует явного создания новых признаков, что неэффективно при больших данных. Для нелинейных задач лучше использовать SVM с ядрами, деревья решений или нейросети.

---
## 8. Кривые валидации и обучения

Мы умеем строить и обучать модели. Но что делать, если качество модели нас не устраивает?

- Усложнить или упростить модель?
- Добавить признаки?
- Нужно просто больше данных?

Ответ на эти вопросы дают **кривые валидации** и **кривые обучения**.

### 8.1 Кривые валидации

Показывают, как качество на обучении и на валидации зависит от **сложности модели** (гиперпараметра).

Работаем с уже знакомым датасетом телеком-оператора.

In [ ]:
data_churn = pd.read_csv("../../data/telecom_churn.csv").drop("State", axis=1)
data_churn["International plan"] = data_churn["International plan"].map({"Yes": 1, "No": 0})
data_churn["Voice mail plan"]     = data_churn["Voice mail plan"].map({"Yes": 1, "No": 0})

y_churn = data_churn["Churn"].astype("int").values
X_churn = data_churn.drop("Churn", axis=1).values

In [ ]:
def plot_with_err(x, data, **kwargs):
    """Строит среднюю линию с полосой ±std."""
    mu, std = data.mean(1), data.std(1)
    lines = plt.plot(x, mu, '-', **kwargs)
    plt.fill_between(x, mu - std, mu + std,
                     edgecolor='none', facecolor=lines[0].get_color(), alpha=0.2)

alphas = np.logspace(-2, 0, 20)
sgd_logit = SGDClassifier(loss="log_loss", n_jobs=-1, random_state=17, max_iter=100)
logit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly",   PolynomialFeatures(degree=2)),
    ("sgd_logit", sgd_logit),
])
val_train, val_test = validation_curve(
    logit_pipe, X_churn, y_churn,
    param_name="sgd_logit__alpha", param_range=alphas,
    cv=5, scoring="roc_auc"
)

In [ ]:
plt.figure(figsize=(8, 4))
plot_with_err(alphas, val_train, label="Обучение")
plot_with_err(alphas, val_test,  label="Валидация (CV)")
plt.xlabel(r"$\alpha$ (сила регуляризации)")
plt.ylabel("ROC-AUC")
plt.title("Кривые валидации")
plt.legend()
plt.grid(True, alpha=0.3)

**Как читать кривые валидации:**

- Кривые **близки и обе низкие** → **недообучение** (underfitting): модель слишком простая.
- Кривые **далеко друг от друга** → **переобучение** (overfitting): модель слишком сложная, «запомнила» шум.
- Кривые **близки и обе высокие** → хороший баланс.

### 8.2 Кривые обучения

Показывают, как качество зависит от **объёма обучающих данных**. Помогают ответить: «Поможет ли сбор дополнительных данных?»

In [ ]:
def plot_learning_curve(alpha=0.01, degree=2):
    train_sizes = np.linspace(0.05, 1, 20)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly",   PolynomialFeatures(degree=degree)),
        ("sgd_logit", SGDClassifier(loss="log_loss", n_jobs=-1,
                                     random_state=17, alpha=alpha, max_iter=100)),
    ])
    N_train, lc_train, lc_test = learning_curve(
        pipe, X_churn, y_churn,
        train_sizes=train_sizes, cv=5, scoring="roc_auc"
    )
    plot_with_err(N_train, lc_train, label="Обучение")
    plot_with_err(N_train, lc_test,  label="Валидация (CV)")
    plt.xlabel("Размер обучающей выборки")
    plt.ylabel("ROC-AUC")
    plt.legend()
    plt.grid(True, alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cases = [
    (10,    "alpha=10 (недообучение)"),
    (0.05,  "alpha=0.05 (хороший баланс)"),
    (1e-4,  "alpha=0.0001 (переобучение)"),
]

for ax, (alpha, title) in zip(axes, cases):
    plt.sca(ax)
    plot_learning_curve(alpha=alpha)
    ax.set_title(title)

plt.tight_layout()

**Как читать кривые обучения:**

| Картина | Диагноз | Что делать |
|---|---|---|
| Кривые сошлись, обе невысокие | Недообучение | Усложнить модель, добавить признаки |
| Кривые сходятся, зазор уменьшается | Всё хорошо | Добавление данных поможет |
| Кривые не сходятся, большой зазор | Переобучение | Регуляризовать, упростить, собрать больше данных |

**Ключевой вывод:** если кривые уже сошлись, добавление новых данных не поможет. Нужно менять сложность модели.

### Итоги по диагностике модели

- Ошибка на обучающей выборке сама по себе ничего не говорит о качестве модели.
- **Кривая валидации**: качество vs. гиперпараметр → диагностирует недо-/переобучение.
- **Кривая обучения**: качество vs. объём данных → отвечает на вопрос «нужно ли больше данных?»
- Всегда смотрите на **обе кривые** вместе.

---
## 9. Практика

Практические задания используют датасет телеком-оператора, с которым мы уже работали.

In [ ]:
# Данные уже загружены выше как data_churn, X_churn, y_churn
print(f"Объектов: {X_churn.shape[0]}, признаков: {X_churn.shape[1]}")
print(f"Доля оттока: {y_churn.mean():.2%}")

### Задание 1
Обучите `LogisticRegression` (с `StandardScaler` в пайплайне) на данных телеком-оператора.

Подберите оптимальный параметр `C` с помощью `GridSearchCV` (используйте диапазон `np.logspace(-3, 3, 20)`, `cv=5`, метрику `roc_auc`).

Выведите лучшее `C` и соответствующий ROC-AUC.

In [ ]:
# Ваш код здесь

### Задание 2
Постройте кривую валидации для `LogisticRegression`: как ROC-AUC на обучении и кросс-валидации зависит от `C`.

Используйте функцию `validation_curve` из sklearn и `plot_with_err` из ноутбука.

Какое значение C оптимально по кривой?

In [ ]:
# Ваш код здесь

### Задание 3
Постройте кривую обучения для лучшей модели (лучший `C` из задания 1).

Ответьте на вопрос: поможет ли сбор дополнительных данных улучшить качество модели?

In [ ]:
# Ваш код здесь

### Задание 4 (повышенная сложность)
Добавьте полиномиальные признаки степени 2 в пайплайн (после `StandardScaler`, перед `LogisticRegression`).

Подберите `C` и сравните качество с моделью без полиномиальных признаков.

Как изменились кривые обучения? Стала ли модель переобучаться?

In [ ]:
# Ваш код здесь

---
## Полезные ресурсы

- [Документация sklearn: LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Документация sklearn: Ridge, Lasso](https://scikit-learn.org/stable/modules/linear_model.html)
- [validation_curve](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html) и [learning_curve](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html)
- Книга: [«Deep Learning»](http://www.deeplearningbook.org) (Goodfellow, Bengio, Courville) — хороший обзор линейных моделей
- Книга: [«The Elements of Statistical Learning»](https://hastie.su.domains/ElemStatLearn/) (Hastie, Tibshirani, Friedman) — статистическая перспектива